In [ ]:
# Cell 1: Setup
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

if not os.getenv("ANTHROPIC_API_KEY"):
    raise EnvironmentError("ANTHROPIC_API_KEY is not set. Copy .env.example to .env and fill in your key.")

from graph import debate_graph
from memory import upsert_debate, retrieve_context

print("Setup complete. Model:", os.getenv("MODEL_NAME", "claude-haiku-4-5-20251001"))

In [ ]:
# Cell 2: Set debate topic
topic = "Should AI be used in hiring?"
print(f"Topic: {topic}")

In [ ]:
# Cell 3: Preview memory context
context = retrieve_context(topic)

if context:
    display(Markdown("### Memory Context (retrieved for this topic)"))
    for i, c in enumerate(context, 1):
        display(Markdown(f"**Past debate {i}:**\n> {c}"))
else:
    display(Markdown("_No past debates found. Agents will argue from scratch._"))

In [ ]:
# Cell 4: Run the debate
# Uses astream (state-based) instead of astream_events — each node emits a state
# snapshot after it completes, which display() can render reliably in Jupyter.
from IPython.display import display, Markdown
import time

NODE_LABELS = {
    "pro_opening":        "Pro Agent — Opening Round",
    "con_opening":        "Con Agent — Opening Round",
    "pro_rebuttal":       "Pro Agent — Rebuttal",
    "con_rebuttal":       "Con Agent — Rebuttal",
    "pro_closing":        "Pro Agent — Closing Remarks",
    "con_closing":        "Con Agent — Closing Remarks",
    "moderator_decision": "Moderator — Final Decision",
}

NODE_CONTENT_KEY = {
    "pro_opening":        "pro_opening",
    "con_opening":        "con_opening",
    "pro_rebuttal":       "pro_rebuttal",
    "con_rebuttal":       "con_rebuttal",
    "pro_closing":        "pro_closing",
    "con_closing":        "con_closing",
    "moderator_decision": "moderator_summary",
}

initial_state = {
    "topic": topic, "round": "opening", "pro_opening": "", "con_opening": "",
    "pro_rebuttal": "", "con_rebuttal": "", "pro_closing": "", "con_closing": "",
    "moderator_summary": "", "winner": "", "memory_context": [],
}

# Accumulate all updates into final_state so Cell 5/6 work without a second LLM run
final_state = dict(initial_state)

async def run_debate():
    global final_state
    t_total = time.time()

    # stream_mode="updates" yields {node_name: state_delta} after each node completes
    async for chunk in debate_graph.astream(initial_state, stream_mode="updates"):
        for node, update in chunk.items():
            final_state.update(update)   # merge delta into accumulated state

            label = NODE_LABELS.get(node)
            content_key = NODE_CONTENT_KEY.get(node)

            if label and content_key:
                content = update.get(content_key, "")
                print(f"[LOG] {node} completed — {len(content)} chars")
                display(Markdown(f"---\n### {label}\n"))
                display(Markdown(content))
            else:
                print(f"[LOG] {node} → {update}")

    print(f"\n[LOG] Total debate time: {time.time()-t_total:.1f}s")
    print(f"[LOG] Winner: {final_state.get('winner', '(not set)')}")

await run_debate()

In [ ]:
# Cell 5: Winner declaration
display(Markdown("---"))
display(Markdown(f"## Winner: {final_state['winner']}"))
display(Markdown(f"**Moderator's Justification:**\n\n{final_state['moderator_summary']}"))

In [ ]:
# Cell 6: Save debate to memory
upsert_debate(final_state)

display(Markdown("---"))
display(Markdown("**Debate saved to memory.**"))
display(Markdown(
    f"- **Topic:** {final_state['topic']}  \n"
    f"- **Winner:** {final_state['winner']}  \n"
    f"- **Pro excerpt:** {final_state['pro_opening'][:100]}\u2026  \n"
    f"- **Con excerpt:** {final_state['con_opening'][:100]}\u2026"
))

## How to demonstrate memory

1. Go back to **Cell 2** and change `topic` to a related topic (e.g., `"Should AI replace recruiters?"`)
2. Re-run cells **2 → 6** in order
3. In **Cell 3** you will now see the first debate retrieved as context
4. Notice how Pro and Con arguments in Cell 4 evolve beyond the generic points made in the first debate